# item, detach, clone et gel : choisir ce qu’on conserve

Conserver une mesure ou une copie sans changer ce que le modèle apprend.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#detacher-geler-journaliser).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Pour tracer une courbe, nous voulons un nombre. Pour examiner plusieurs prédictions, nous voulons encore un tenseur. Pour modifier ces prédictions sans toucher l’original, il faut aussi une copie.

Reprenons la perte quadratique : avec un poids de 1 et une cible de 2, elle vaut 1. Le tableau [perte,2×perte] vaut donc [1,2].

| Besoin | Opération | Résultat |
| --- | --- | --- |
| Un nombre Python | perte.item() | 1.0, sans méthode backward |
| Un tenseur sans lien au calcul précédent | valeurs.detach() | Même valeur, graphe coupé |
| Un stockage indépendant | tenseur.clone() | Valeurs copiées, graphe éventuellement conservé |

item extrait une seule valeur. detach et clone répondent à des besoins différents ; nous les combinons quand les deux propriétés sont nécessaires.

## Démonstration 1 · Voir l’original et la copie côte à côte

Vérifier la valeur, le type et la présence du lien autograd.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

poids est le paramètre à ajuster. perte reste reliée à ce poids, même si sa valeur numérique est simplement 1.

In [4]:
poids = torch.tensor(1., requires_grad=True)
perte = (poids - 2.) ** 2


In [5]:
observer('poids', poids)
observer('perte', perte)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
perte forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Calculer l’état suivant

item() renvoie un float Python. nombre sert au journal ; perte reste disponible pour backward.

In [6]:
nombre = perte.item()


In [7]:
observer('nombre', nombre)

nombre = 1.0


### Calculer l’état suivant

stack réunit deux scalaires en un vecteur. detach donne accès à ses valeurs sans continuer la dérivation depuis ce résultat.

In [8]:
valeurs = torch.stack([perte, 2 * perte])
detachees = valeurs.detach()


In [9]:
observer('valeurs', valeurs)
observer('detachees', detachees)

valeurs forme (2,) type torch.float32 requires_grad True
  valeurs : [1.0, 2.0]
detachees forme (2,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0]


### Calculer l’état suivant

clone copie le stockage. copie provient déjà du tenseur détaché ; sa première valeur peut devenir 99 sans modifier valeurs.

In [10]:
copie = detachees.clone()
copie[0] = 99.


In [11]:
observer('copie', copie)

copie forme (2,) type torch.float32 requires_grad False
  valeurs : [99.0, 2.0]


### Calculer l’état suivant

Un clone direct de valeurs conserve le lien autograd. Le booléen requires_grad vérifie donc une propriété différente de l’indépendance du stockage.

In [12]:
copie_avec_graphe = valeurs.clone()


In [13]:
observer('copie_avec_graphe', copie_avec_graphe)

copie_avec_graphe forme (2,) type torch.float32 requires_grad True
  valeurs : [1.0, 2.0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("nombre :", nombre, type(nombre).__name__)
print("original :", valeurs.tolist())
print("copie modifiee :", copie.tolist())
print("gradients :", valeurs.requires_grad, detachees.requires_grad,
      copie.requires_grad, copie_avec_graphe.requires_grad)


nombre : 1.0 float
original : [1.0, 2.0]
copie modifiee : [99.0, 2.0]
gradients : True False False True


### Calculer les gradients

Nous appelons backward sur perte, pas sur nombre. La dérivée de (poids−2)² au point 1 vaut −2.

In [15]:
perte.backward()


In [16]:
observer_entrainement({'poids': poids, 'perte': perte, 'nombre': nombre, 'valeurs': valeurs, 'detachees': detachees, 'copie': copie, 'copie_avec_graphe': copie_avec_graphe}, montrer_poids=False)

poids.grad forme () type torch.float32 requires_grad False
  valeurs : -2.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [17]:
print("gradient du poids :", poids.grad.item())

gradient du poids : -2.0


Journaliser n’a pas remplacé la perte originale. Nous pouvons toujours calculer son gradient.

detach ne copie pas nécessairement le stockage. Modifier le tenseur seulement détaché peut donc modifier son original ; detach().clone() évite ce partage.

### Autre besoin : garder un encodeur fixe

Nous revenons aux trois extraits fictifs et à leurs classes. Un encodeur produit ici deux coordonnées intermédiaires ; seule la tête de classification doit apprendre.

requires_grad_(False) désactive les gradients des paramètres de l’encodeur. eval() règle séparément le comportement de ses couches. Ce petit exemple ne contient aucune couche sensible au mode.

## Démonstration 2 · Vérifier où les gradients apparaissent

Comparer l’encodeur gelé à la tête qui reste entraînable.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [18]:
import torch
from torch import nn


### Calculer l’état suivant

Les données sont de forme (3,2). Les deux couches Linear ont deux entrées et deux sorties ; leurs poids sont initialisés aléatoirement.

In [19]:
torch.manual_seed(17)
entrees = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
encodeur = nn.Linear(2, 2)


In [20]:
observer('entrees', entrees)
observer('cibles', cibles)

entrees forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [21]:
tete = nn.Linear(2, 2)


In [22]:
observer('tete', tete)

### Répéter la même étape sur les données

La boucle parcourt poids et biais de l’encodeur. requires_grad_(False) agit sur ces paramètres ; eval ne réalise pas ce gel.

In [23]:
for parametre in encodeur.parameters():
    parametre.requires_grad_(False)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [24]:
encodeur.eval()


Linear(in_features=2, out_features=2, bias=True)

### Calculer l’état suivant

L’encodeur gelé ne demande pas de gradient et les entrées non plus. Sa représentation n’a donc pas de graphe ; la tête en construit un nouveau.

In [25]:
representation = encodeur(entrees)
scores = tete(representation)
critere = nn.CrossEntropyLoss()
perte = critere(scores, cibles)


In [26]:
observer('representation', representation)
observer('scores', scores)
observer('perte', perte)

representation forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[-0.6004615426063538, -0.6813506484031677], [-0.4608038663864136, -2.230463743209839], [-0.5554589033126831, -1.1899385452270508]]
scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[-0.6216570734977722, 0.30329629778862], [-0.7462103962898254, 1.354841709136963], [-0.6624081134796143, 0.6485661864280701]]
perte forme () type torch.float32 requires_grad True
  valeurs : 0.9746685028076172


### Calculer les gradients

Pour l’encodeur, nous testons grad is None : True signifie gradient absent. Pour la tête, le test inverse confirme un gradient présent.

In [27]:
perte.backward()


In [28]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'encodeur': encodeur, 'tete': tete, 'representation': representation, 'scores': scores, 'critere': critere, 'perte': perte}, montrer_poids=False)

encodeur.weight.grad = None
encodeur.bias.grad = None
tete.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.2724190950393677, 0.3940196931362152], [-0.2724190950393677, -0.3940197229385376]]
tete.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [-0.464910626411438, 0.4649106562137604]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [29]:
print("graphe de la representation :", representation.requires_grad)
print("gradients encodeur :", [p.grad is None for p in encodeur.parameters()])
print("gradients tete :", [p.grad is not None for p in tete.parameters()])

graphe de la representation : False
gradients encodeur : [True, True]
gradients tete : [True, True]


Geler une partie ne dispense pas de définir quelle autre partie apprend et comment elle reçoit ses entrées.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Nous modifions la première case d’un tenseur détaché. Que vaut la première case de la source ?

In [30]:
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_detache = exo_x.detach()
exo_detache[0] = 9.
exo_source = ...  # Remplacer les points par votre réponse.
if exo_source is Ellipsis or exo_source is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_source)-exo_x[0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

detach retire le lien au graphe, mais partage encore le stockage.

</details>

### Exercice 2

Construisez cette fois une copie détachée indépendante avant de modifier sa première case.

In [31]:
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_copie = ...  # Remplacer les points par votre réponse.
if exo_copie is Ellipsis or exo_copie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_copie[0]=9.
    assert exo_x[0].item() == 1.
    assert not exo_copie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Deux propriétés sont demandées : aucun lien au graphe et aucun stockage partagé.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_detache = exo_x.detach()
exo_detache[0] = 9.
exo_source = 9.
if exo_source is Ellipsis or exo_source is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_source)-exo_x[0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_copie = exo_x.detach().clone()
if exo_copie is Ellipsis or exo_copie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_copie[0]=9.
    assert exo_x[0].item() == 1.
    assert not exo_copie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [32]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()